In [ ]:
from pathlib import Path
import json
import random
from collections import Counter, defaultdict
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

In [2]:
REPO_ROOT = Path("/scratch/workspace/emma/standard-project-m2-mystery-machine")

PROCESSED_DIR = REPO_ROOT / "multilingual" / "data" / "processed"
DISTILLED_ROOT = REPO_ROOT / "multilingual" / "data" / "distilled" / "v2_fallback"

BENCHMARK_DIR = REPO_ROOT / "multilingual" / "data" / "benchmarks"
BY_LANGUAGE_DIR = BENCHMARK_DIR / "by_language"
BY_DATASET_DIR = BENCHMARK_DIR / "by_dataset"

for p in [BENCHMARK_DIR, BY_LANGUAGE_DIR, BY_DATASET_DIR]:
    p.mkdir(parents=True, exist_ok=True)

ALL_PATH = BENCHMARK_DIR / "benchmark_all.jsonl"
MANIFEST_PATH = BENCHMARK_DIR / "manifest.json"

RANDOM_SEED = 42
VALID_ANSWER_LETTERS = set("ABCDEFGHIJKLMNOPQRSTUVWXYZ")

print("REPO_ROOT:      ", REPO_ROOT)
print("PROCESSED_DIR:  ", PROCESSED_DIR)
print("DISTILLED_ROOT: ", DISTILLED_ROOT)
print("BENCHMARK_DIR:  ", BENCHMARK_DIR)

REPO_ROOT:       /scratch/workspace/emma/standard-project-m2-mystery-machine
PROCESSED_DIR:   /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed
DISTILLED_ROOT:  /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/distilled/v2_fallback
BENCHMARK_DIR:   /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks


# Helpers

In [3]:
def read_jsonl(path):
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except Exception as e:
                raise ValueError(f"Bad JSON in {path}:{line_no}: {e}") from e
    return records


def write_jsonl(records, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def get_source_id(record):
    sid = record.get("source_id")
    if sid is None or str(sid).strip() == "":
        sid = record.get("id")
    if sid is None:
        return None
    return str(sid).strip()


def normalize_text(x):
    if x is None:
        return ""
    return str(x).strip()


def safe_name(x):
    return (
        str(x)
        .strip()
        .replace("/", "_")
        .replace("\\", "_")
        .replace(" ", "_")
        .replace("-", "_")
        .replace(":", "_")
    )


def validate_basic_record(record, require_answer=True):
    record_id = record.get("id") or record.get("source_id")

    assert isinstance(record_id, str) and record_id.strip()
    assert isinstance(record.get("dataset"), str) and record["dataset"].strip()
    assert record.get("language") in {"russian", "chinese", "italian", "spanish", "hindi"}
    assert record.get("task_type") == "mcq"
    assert isinstance(record.get("prompt"), str) and record["prompt"].strip()
    assert isinstance(record.get("metadata"), dict)

    if require_answer:
        answer = str(record.get("answer", "")).strip().upper()
        assert answer in VALID_ANSWER_LETTERS
        assert f"{answer}) " in record["prompt"], record_id

In [ ]:
CANDIDATE_FILES = {
    # INCLUDE
    ("include", "russian"): [PROCESSED_DIR / "include" / "include_russian_all.jsonl"],
    ("include", "chinese"): [PROCESSED_DIR / "include" / "include_chinese_all.jsonl"],
    ("include", "italian"): [PROCESSED_DIR / "include" / "include_italian_all.jsonl"],
    ("include", "spanish"): [PROCESSED_DIR / "include" / "include_spanish_all.jsonl"],
    ("include", "hindi"): [PROCESSED_DIR / "include" / "include_hindi_all.jsonl"],

    # Global MMLU
    ("global_mmlu", "russian"): [PROCESSED_DIR / "global_mmlu" / "global_mmlu_russian_all.jsonl"],
    ("global_mmlu", "chinese"): [PROCESSED_DIR / "global_mmlu" / "global_mmlu_chinese_all.jsonl"],
    ("global_mmlu", "italian"): [PROCESSED_DIR / "global_mmlu" / "global_mmlu_italian_all.jsonl"],
    ("global_mmlu", "spanish"): [PROCESSED_DIR / "global_mmlu" / "global_mmlu_spanish_all.jsonl"],
    ("global_mmlu", "hindi"): [PROCESSED_DIR / "global_mmlu" / "global_mmlu_hindi_all.jsonl"],

    # Language-specific datasets
    ("ceval", "chinese"): [PROCESSED_DIR / "ceval" / "ceval_chinese_all.jsonl"],
    ("cmmlu", "chinese"): [PROCESSED_DIR / "cmmlu" / "cmmlu_chinese_eval.jsonl"],
    ("milu", "hindi"): [PROCESSED_DIR / "milu" / "milu_hindi_all.jsonl"],
    # ("exams", "italian"): [PROCESSED_DIR / "exams" / "exams_italian_eval.jsonl"],
    ("head_qa", "spanish"): [PROCESSED_DIR / "head_qa" / "head_qa_spanish_eval.jsonl"],
    ("mera", "russian"): [PROCESSED_DIR / "mera" / "mera_clean_all.jsonl"],
}

missing = []
empty_groups = []

for key, paths in CANDIDATE_FILES.items():
    if not paths:
        empty_groups.append(key)
        continue

    for path in paths:
        if not path.exists():
            missing.append((key, path))

print("Empty groups:")
for key in empty_groups:
    print(key)

print("\nMissing files:")
for key, path in missing:
    print(key, path)

print("\nCandidate file groups:")
for key, paths in CANDIDATE_FILES.items():
    print(key)
    for path in paths:
        print("  ", path.relative_to(REPO_ROOT))

assert not empty_groups, "Some candidate groups have no files."
assert not missing, "Some candidate files are missing. Fix CANDIDATE_FILES paths before continuing."

Empty groups:

Missing files:

Candidate file groups:
('include', 'russian')
   multilingual/data/processed/include/include_russian_all.jsonl
('include', 'chinese')
   multilingual/data/processed/include/include_chinese_all.jsonl
('include', 'italian')
   multilingual/data/processed/include/include_italian_all.jsonl
('include', 'spanish')
   multilingual/data/processed/include/include_spanish_all.jsonl
('include', 'hindi')
   multilingual/data/processed/include/include_hindi_all.jsonl
('global_mmlu', 'russian')
   multilingual/data/processed/global_mmlu/global_mmlu_russian_all.jsonl
('global_mmlu', 'chinese')
   multilingual/data/processed/global_mmlu/global_mmlu_chinese_all.jsonl
('global_mmlu', 'italian')
   multilingual/data/processed/global_mmlu/global_mmlu_italian_all.jsonl
('global_mmlu', 'spanish')
   multilingual/data/processed/global_mmlu/global_mmlu_spanish_all.jsonl
('global_mmlu', 'hindi')
   multilingual/data/processed/global_mmlu/global_mmlu_hindi_all.jsonl
('ceval', 'chi

In [5]:
candidate_records = {}
candidate_rejected_records = []

for key, paths in CANDIDATE_FILES.items():
    expected_dataset, expected_language = key
    records = []

    for path in paths:
        file_records = read_jsonl(path)

        for source_row_idx, r in enumerate(file_records):
            try:
                # Validate schema, allowing id fallback and variable answer letters.
                validate_basic_record(r, require_answer=False)

                assert r["language"] == expected_language, (
                    path,
                    r.get("language"),
                    expected_language,
                )

                r = dict(r)

                # Repair missing top-level id from source_id.
                if not isinstance(r.get("id"), str) or not r["id"].strip():
                    fallback_id = r.get("source_id")
                    assert isinstance(fallback_id, str) and fallback_id.strip(), (
                        path,
                        source_row_idx,
                        r.get("id"),
                        r.get("source_id"),
                    )
                    r["id"] = fallback_id

                # Normalize MERA dataset labels.
                if expected_dataset == "mera":
                    assert str(r["dataset"]).startswith("mera"), (
                        path,
                        r["dataset"],
                        expected_dataset,
                    )
                    r["dataset"] = "mera"
                else:
                    assert r["dataset"] == expected_dataset, (
                        path,
                        r["dataset"],
                        expected_dataset,
                    )

                answer = str(r.get("answer", "")).strip().upper()
                prompt = str(r.get("prompt", ""))

                if answer not in VALID_ANSWER_LETTERS:
                    candidate_rejected_records.append({
                        "reason": "invalid_answer_letter",
                        "key": key,
                        "path": str(path.relative_to(REPO_ROOT)),
                        "source_row_idx": source_row_idx,
                        "id": r.get("id", ""),
                        "source_id": r.get("source_id", ""),
                        "answer": r.get("answer", ""),
                    })
                    continue

                if f"{answer}) " not in prompt:
                    candidate_rejected_records.append({
                        "reason": "answer_option_not_in_prompt",
                        "key": key,
                        "path": str(path.relative_to(REPO_ROOT)),
                        "source_row_idx": source_row_idx,
                        "id": r.get("id", ""),
                        "source_id": r.get("source_id", ""),
                        "answer": r.get("answer", ""),
                    })
                    continue

                r["answer"] = answer
                records.append(r)

            except Exception as e:
                candidate_rejected_records.append({
                    "reason": repr(e),
                    "key": key,
                    "path": str(path.relative_to(REPO_ROOT)),
                    "source_row_idx": source_row_idx,
                    "id": r.get("id", ""),
                    "source_id": r.get("source_id", ""),
                    "answer": r.get("answer", ""),
                })

    candidate_records[key] = records
    print(f"{key}: {len(records)} valid records from {len(paths)} file(s)")

rejected_df = pd.DataFrame(candidate_rejected_records)

print(f"\nRejected candidate records: {len(rejected_df)}")
if len(rejected_df):
    display(rejected_df.head(50))
    display(
        rejected_df
        .groupby(["key", "reason"])
        .size()
        .reset_index(name="n")
        .sort_values(["key", "n"], ascending=[True, False])
    )

('include', 'russian'): 586 valid records from 1 file(s)
('include', 'chinese'): 584 valid records from 1 file(s)
('include', 'italian'): 575 valid records from 1 file(s)
('include', 'spanish'): 570 valid records from 1 file(s)
('include', 'hindi'): 584 valid records from 1 file(s)
('global_mmlu', 'russian'): 13974 valid records from 1 file(s)
('global_mmlu', 'chinese'): 13964 valid records from 1 file(s)
('global_mmlu', 'italian'): 13935 valid records from 1 file(s)
('global_mmlu', 'spanish'): 13976 valid records from 1 file(s)
('global_mmlu', 'hindi'): 13991 valid records from 1 file(s)
('ceval', 'chinese'): 13945 valid records from 1 file(s)
('cmmlu', 'chinese'): 335 valid records from 1 file(s)
('milu', 'hindi'): 15622 valid records from 1 file(s)
('exams', 'italian'): 200 valid records from 1 file(s)
('head_qa', 'spanish'): 200 valid records from 1 file(s)
('mera', 'russian'): 3918 valid records from 1 file(s)

Rejected candidate records: 0


In [6]:
summary_rows = []

for (dataset, language), records in candidate_records.items():
    ids = [get_source_id(r) for r in records]
    subjects = [
        r.get("metadata", {}).get("subject", "")
        or r.get("metadata", {}).get("subset", "")
        or r.get("metadata", {}).get("category", "")
        or r.get("metadata", {}).get("task", "")
        or r.get("subset", "")
        or ""
        for r in records
    ]

    summary_rows.append({
        "dataset": dataset,
        "language": language,
        "n_records": len(records),
        "n_unique_source_ids": len(set(ids)),
        "n_missing_source_ids": sum(1 for x in ids if not x),
        "n_subjects_or_categories": len(set(s for s in subjects if s)),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(["language", "dataset"])
display(summary_df)

,dataset,language,n_records,n_unique_source_ids,n_missing_source_ids,n_subjects_or_categories
10,ceval,chinese,13945,13945,0,52
11,cmmlu,chinese,335,335,0,67
6,global_mmlu,chinese,13964,13964,0,57
1,include,chinese,584,584,0,32
9,global_mmlu,hindi,13991,13991,0,57
4,include,hindi,584,584,0,18
12,milu,hindi,15622,15622,0,41
13,exams,italian,200,200,0,12
7,global_mmlu,italian,13935,13935,0,57
2,include,italian,575,575,0,19


In [7]:
active_distilled_paths = sorted(
    p for p in DISTILLED_ROOT.glob("**/*_distilled.jsonl")
    if "backup_before_benchmark_filter_" not in str(p)
)

print(f"Active distilled files: {len(active_distilled_paths)}")

for p in active_distilled_paths[:20]:
    print(p.relative_to(REPO_ROOT))

if len(active_distilled_paths) > 20:
    print("...")

distilled_ids = set()

for path in active_distilled_paths:
    for r in read_jsonl(path):
        sid = get_source_id(r)
        if sid:
            distilled_ids.add(sid)

print("Unique active distilled source IDs:", len(distilled_ids))

Active distilled files: 77
multilingual/data/distilled/v2_fallback/ceval_chinese_regional_distilled.jsonl
multilingual/data/distilled/v2_fallback/ceval_chinese_regional_implicit_distilled.jsonl
multilingual/data/distilled/v2_fallback/cultsportqa_chinese_distilled.jsonl
multilingual/data/distilled/v2_fallback/cultsportqa_hindi_distilled.jsonl
multilingual/data/distilled/v2_fallback/exams_italian_distilled.jsonl
multilingual/data/distilled/v2_fallback/exams_spanish_distilled.jsonl
multilingual/data/distilled/v2_fallback/global_mmlu_chinese_region_agnostic_distilled.jsonl
multilingual/data/distilled/v2_fallback/global_mmlu_chinese_regional_distilled.jsonl
multilingual/data/distilled/v2_fallback/global_mmlu_hindi_region_agnostic_distilled.jsonl
multilingual/data/distilled/v2_fallback/global_mmlu_hindi_regional_distilled.jsonl
multilingual/data/distilled/v2_fallback/global_mmlu_italian_region_agnostic_distilled.jsonl
multilingual/data/distilled/v2_fallback/global_mmlu_italian_regional_disti

In [8]:
contam_rows = []

for (dataset, language), records in candidate_records.items():
    ids = [get_source_id(r) for r in records if get_source_id(r)]
    unique_ids = set(ids)
    overlap = unique_ids & distilled_ids

    contam_rows.append({
        "dataset": dataset,
        "language": language,
        "n_records": len(records),
        "n_unique_ids": len(unique_ids),
        "n_overlap_with_active_distilled": len(overlap),
        "n_safe_candidates": len(unique_ids - distilled_ids),
    })

contam_df = pd.DataFrame(contam_rows).sort_values(["language", "dataset"])
display(contam_df)

,dataset,language,n_records,n_unique_ids,n_overlap_with_active_distilled,n_safe_candidates
10,ceval,chinese,13945,13945,5945,8000
11,cmmlu,chinese,335,335,0,335
6,global_mmlu,chinese,13964,13964,4001,9963
1,include,chinese,584,584,383,201
9,global_mmlu,hindi,13991,13991,3824,10167
4,include,hindi,584,584,361,223
12,milu,hindi,15622,15622,6702,8920
13,exams,italian,200,200,0,200
7,global_mmlu,italian,13935,13935,3987,9948
2,include,italian,575,575,428,147


In [9]:
TARGET_COUNTS = {
    "russian": {
        "include": 100,
        "global_mmlu": 200,
        "mera": 200,
    },
    "chinese": {
        "include": 100,
        "global_mmlu": 100,
        "ceval": 150,
        "cmmlu": 150,
    },
    "hindi": {
        "include": 100,
        "global_mmlu": 200,
        "milu": 200,
    },
    "italian": {
        "include": 100,
        "global_mmlu": 200,
        "exams": 200,
    },
    "spanish": {
        "include": 100,
        "global_mmlu": 200,
        "head_qa": 200,
    },
}

sum_by_lang = {lang: sum(d.values()) for lang, d in TARGET_COUNTS.items()}
sum_total = sum(sum_by_lang.values())

print("Target by language:", sum_by_lang)
print("Target total:", sum_total)

assert all(n == 500 for n in sum_by_lang.values())
assert sum_total == 2500

Target by language: {'russian': 500, 'chinese': 500, 'hindi': 500, 'italian': 500, 'spanish': 500}
Target total: 2500


In [10]:
def record_subject(record):
    meta = record.get("metadata", {})
    return (
        meta.get("subject")
        or meta.get("subset")
        or meta.get("task")
        or meta.get("source_task")
        or meta.get("category")
        or meta.get("dataset_name")
        or record.get("subset")
        or record.get("dataset")
        or "unknown"
    )


def filter_safe(records):
    safe = []

    for record in records:
        source_id = get_source_id(record)

        if source_id and source_id not in distilled_ids:
            safe.append(record)

    return safe


def sample_uniform(records, n, seed):
    if len(records) < n:
        raise ValueError(f"Need {n} records but only have {len(records)} candidates.")

    if len(records) == n:
        return list(records)

    rng = random.Random(seed)
    return rng.sample(list(records), n)


def sample_subject_stratified(records, n, seed):
    if len(records) < n:
        raise ValueError(f"Need {n} records but only have {len(records)} candidates.")

    by_subject = defaultdict(list)

    for record in records:
        subject = record_subject(record)
        by_subject[subject].append(record)

    rng = random.Random(seed)

    subjects = sorted(by_subject)
    for subject in subjects:
        rng.shuffle(by_subject[subject])

    selected = []
    selected_ids = set()

    while len(selected) < n:
        made_progress = False

        for subject in subjects:
            if len(selected) >= n:
                break

            bucket = by_subject[subject]

            while bucket:
                candidate = bucket.pop()
                source_id = get_source_id(candidate)

                if source_id not in selected_ids:
                    selected.append(candidate)
                    selected_ids.add(source_id)
                    made_progress = True
                    break

        if not made_progress:
            break

    if len(selected) < n:
        raise ValueError(f"Only selected {len(selected)} records, need {n}.")

    return selected

In [11]:
selected_components = {}

STRATIFIED_DATASETS = {
    "global_mmlu",
    "mera",
    "ceval",
    "milu",
    "cmmlu",
}

for language, dataset_counts in TARGET_COUNTS.items():
    for dataset, n in dataset_counts.items():
        key = (dataset, language)

        if key not in candidate_records:
            raise KeyError(f"Missing candidate records for {key}")

        records = candidate_records[key]
        safe_records = filter_safe(records)

        seed = RANDOM_SEED + abs(hash((dataset, language))) % 10_000

        if dataset in STRATIFIED_DATASETS:
            selected = sample_subject_stratified(safe_records, n, seed)
        else:
            selected = sample_uniform(safe_records, n, seed)

        selected_components[key] = selected

        print(
            f"{language:8s} | {dataset:12s} | "
            f"selected {len(selected):4d} / safe {len(safe_records):5d} / total {len(records):5d}"
        )

russian  | include      | selected  100 / safe   227 / total   586
russian  | global_mmlu  | selected  200 / safe  9956 / total 13974
russian  | mera         | selected  200 / safe  1408 / total  3918
chinese  | include      | selected  100 / safe   201 / total   584
chinese  | global_mmlu  | selected  100 / safe  9963 / total 13964
chinese  | ceval        | selected  150 / safe  8000 / total 13945
chinese  | cmmlu        | selected  150 / safe   335 / total   335
hindi    | include      | selected  100 / safe   223 / total   584
hindi    | global_mmlu  | selected  200 / safe 10167 / total 13991
hindi    | milu         | selected  200 / safe  8920 / total 15622
italian  | include      | selected  100 / safe   147 / total   575
italian  | global_mmlu  | selected  200 / safe  9948 / total 13935
italian  | exams        | selected  200 / safe   200 / total   200
spanish  | include      | selected  100 / safe   190 / total   570
spanish  | global_mmlu  | selected  200 / safe 10018 / total 1

In [12]:
benchmark_records = []

for (dataset, language), records in selected_components.items():
    for record in records:
        r = dict(record)

        # Repair id if needed
        if not isinstance(r.get("id"), str) or not r["id"].strip():
            r["id"] = get_source_id(r)

        # Canonical benchmark split
        r["split"] = "test"

        # Preserve and enrich metadata
        metadata = dict(r.get("metadata", {}))
        metadata["benchmark_name"] = "cs552_multilingual_regional_mcqa"
        metadata["benchmark_version"] = "v1"
        metadata["benchmark_component"] = dataset
        metadata["benchmark_language"] = language
        metadata["benchmark_source_id"] = get_source_id(r)

        r["metadata"] = metadata

        benchmark_records.append(r)

print("Total benchmark records:", len(benchmark_records))

source_ids = [get_source_id(r) for r in benchmark_records]
assert len(source_ids) == len(set(source_ids)), "Duplicate source IDs in final benchmark."

for r in benchmark_records:
    validate_basic_record(r)
    assert get_source_id(r) not in distilled_ids

print("All benchmark records validated.")

Total benchmark records: 2500
All benchmark records validated.


In [13]:
bench_df = pd.DataFrame(
    {
        "id": r["id"],
        "source_id": get_source_id(r),
        "dataset": r["dataset"],
        "language": r["language"],
        "split": r["split"],
        "answer": str(r["answer"]).strip().upper(),
        "subject": record_subject(r),
        "regionality": r.get("metadata", {}).get("regionality", ""),
    }
    for r in benchmark_records
)

print("Total:", len(bench_df))

print("\nBy language:")
display(bench_df["language"].value_counts().sort_index().to_frame("n"))

print("\nBy dataset:")
display(bench_df["dataset"].value_counts().sort_index().to_frame("n"))

print("\nBy language and dataset:")
display(
    bench_df.groupby(["language", "dataset"])
    .size()
    .reset_index(name="n")
    .pivot(index="language", columns="dataset", values="n")
    .fillna(0)
    .astype(int)
)

print("\nBy answer:")
display(
    bench_df.groupby(["language", "answer"])
    .size()
    .reset_index(name="n")
    .pivot(index="language", columns="answer", values="n")
    .fillna(0)
    .astype(int)
)

Total: 2500

By language:


,n
language,
chinese,500
hindi,500
italian,500
russian,500
spanish,500



By dataset:


,n
dataset,
ceval,150
cmmlu,150
exams,200
global_mmlu,900
head_qa,200
include,500
mera,200
milu,200



By language and dataset:


dataset,ceval,cmmlu,exams,global_mmlu,head_qa,include,mera,milu
language,,,,,,,,
chinese,150,150,0,100,0,100,0,0
hindi,0,0,0,200,0,100,0,200
italian,0,0,200,200,0,100,0,0
russian,0,0,0,200,0,100,200,0
spanish,0,0,0,200,200,100,0,0



By answer:


answer,A,B,C,D,E
language,,,,,
chinese,108,116,136,140,0
hindi,123,127,132,118,0
italian,120,135,119,126,0
russian,132,137,118,113,0
spanish,112,128,135,116,9


In [14]:
print("Top subjects/categories by language and dataset:")
display(
    bench_df.groupby(["language", "dataset", "subject"])
    .size()
    .reset_index(name="n")
    .sort_values(["language", "dataset", "n"], ascending=[True, True, False])
    .head(150)
)

print("\nNumber of subjects/categories per component:")
display(
    bench_df.groupby(["language", "dataset"])["subject"]
    .nunique()
    .reset_index(name="n_subjects")
    .sort_values(["language", "dataset"])
)

Top subjects/categories by language and dataset:


,language,dataset,subject,n
0,chinese,ceval,accountant,3
1,chinese,ceval,advanced_mathematics,3
2,chinese,ceval,art_studies,3
3,chinese,ceval,basic_medicine,3
4,chinese,ceval,business_administration,3
5,chinese,ceval,chinese_language_and_literature,3
6,chinese,ceval,civil_servant,3
7,chinese,ceval,clinical_medicine,3
8,chinese,ceval,college_chemistry,3
9,chinese,ceval,college_economics,3



Number of subjects/categories per component:


,language,dataset,n_subjects
0,chinese,ceval,52
1,chinese,cmmlu,67
2,chinese,global_mmlu,57
3,chinese,include,29
4,hindi,global_mmlu,57
5,hindi,include,16
6,hindi,milu,41
7,italian,exams,12
8,italian,global_mmlu,57
9,italian,include,16


In [15]:
final_overlap = set(bench_df["source_id"]) & distilled_ids

print("Final benchmark source IDs:", bench_df["source_id"].nunique())
print("Overlap with active distilled IDs:", len(final_overlap))

assert len(final_overlap) == 0

Final benchmark source IDs: 2500
Overlap with active distilled IDs: 0


## move old ones before adding headqa/exams/cmmlu

In [16]:
import shutil

OLD_DIR = BENCHMARK_DIR / "old"
OLD_DIR.mkdir(parents=True, exist_ok=True)

# Move only existing flat files that are not the new subfolders / manifest / README.
old_flat_patterns = [
    "benchmark_*.jsonl",
    "cmmlu_chinese_eval.jsonl",
    "exams_italian_eval.jsonl",
    "head_qa_spanish_eval.jsonl",
    "benchmark_distilled_source_id_overlap_report.json",
    "benchmark_distilled_overlapping_source_ids.txt",
]

moved_files = []

for pattern in old_flat_patterns:
    for path in BENCHMARK_DIR.glob(pattern):
        if path.is_file():
            target = OLD_DIR / path.name

            # Avoid overwriting if rerun.
            if target.exists():
                target = OLD_DIR / f"{path.stem}_oldcopy{path.suffix}"

            shutil.move(str(path), str(target))
            moved_files.append((path.name, target.name))

print(f"Moved {len(moved_files)} old files to {OLD_DIR}:")
for src, dst in moved_files:
    print(f"  {src} -> old/{dst}")

Moved 1 old files to /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/old:
  benchmark_all.jsonl -> old/benchmark_all.jsonl


In [17]:
TOP_LEVEL_ORDER = [
    "source_id",
    "language",
    "dataset",
    "prompt",
    "answer",
    "id",
    "split",
    "task_type",
    "metadata",
]

METADATA_PRIORITY_ORDER = [
    "benchmark_name",
    "benchmark_version",
    "benchmark_component",
    "benchmark_language",
    "benchmark_source_id",

    "subject",
    "category",
    "regionality",
    "subject_category",
    "cultural_sensitivity_label",

    "source_split",
    "source_idx",
    "source_path",
    "sample_id",
    "qid",
    "name",
    "year",
    "n_answers",
    "n_choices",
    "subset",
    "task",
]


def reorder_metadata(metadata):
    metadata = dict(metadata or {})
    ordered = {}

    for key in METADATA_PRIORITY_ORDER:
        if key in metadata:
            ordered[key] = metadata[key]

    for key in sorted(metadata.keys()):
        if key not in ordered:
            ordered[key] = metadata[key]

    return ordered


def standardize_benchmark_record(record):
    record = dict(record)

    source_id = get_source_id(record)
    assert source_id

    record_id = record.get("id")
    if not isinstance(record_id, str) or not record_id.strip():
        record_id = source_id

    metadata = reorder_metadata(record.get("metadata", {}))
    metadata["benchmark_source_id"] = source_id

    standardized = {
        "source_id": source_id,
        "language": str(record["language"]).strip(),
        "dataset": str(record["dataset"]).strip(),
        "prompt": str(record["prompt"]).strip(),
        "answer": str(record["answer"]).strip().upper(),
        "id": str(record_id).strip(),
        "split": "test",
        "task_type": "mcq",
        "metadata": metadata,
    }

    return {key: standardized[key] for key in TOP_LEVEL_ORDER}


benchmark_records = [
    standardize_benchmark_record(record)
    for record in benchmark_records
]

bench_df = pd.DataFrame(
    {
        "source_id": r["source_id"],
        "language": r["language"],
        "dataset": r["dataset"],
        "answer": r["answer"],
        "id": r["id"],
        "split": r["split"],
        "subject": record_subject(r),
        "regionality": r.get("metadata", {}).get("regionality", ""),
    }
    for r in benchmark_records
)

print("Standardized benchmark records:", len(benchmark_records))
print("Top-level keys:")
print(list(benchmark_records[0].keys()))

print("\nExample:")
print(json.dumps(benchmark_records[0], ensure_ascii=False, indent=2)[:3000])

Standardized benchmark records: 2500
Top-level keys:
['source_id', 'language', 'dataset', 'prompt', 'answer', 'id', 'split', 'task_type', 'metadata']

Example:
{
  "source_id": "include_russian_all_test_461",
  "language": "russian",
  "dataset": "include",
  "prompt": "Какой из принципов относится к организационно-функциональным?\n\nA) законности;\nB) диспозитивности;\nC) непрерывности судебного разбирательства;\nD) равенства граждан и организаций перед законом и судом.",
  "answer": "D",
  "id": "include_russian_all_test_461",
  "split": "test",
  "task_type": "mcq",
  "metadata": {
    "benchmark_name": "cs552_multilingual_regional_mcqa",
    "benchmark_version": "v1",
    "benchmark_component": "include",
    "benchmark_language": "russian",
    "benchmark_source_id": "include_russian_all_test_461",
    "subject": "Law",
    "regionality": "regional",
    "source_split": "test",
    "source_idx": 461,
    "country": "Russia",
    "domain": "Arts & Humanities",
    "level": "Academi

## write new ones

In [18]:
# Ensure output folders exist
BY_LANGUAGE_DIR.mkdir(parents=True, exist_ok=True)
BY_DATASET_DIR.mkdir(parents=True, exist_ok=True)

# 1. Canonical full benchmark
write_jsonl(benchmark_records, ALL_PATH)
print(f"Saved full benchmark: {len(benchmark_records)} -> {ALL_PATH}")

# 2. By-language files
language_paths = {}

for language in sorted(TARGET_COUNTS):
    language_records = [
        record for record in benchmark_records
        if record["language"] == language
    ]

    assert len(language_records) == 500

    path = BY_LANGUAGE_DIR / f"benchmark_{language}.jsonl"
    write_jsonl(language_records, path)
    language_paths[language] = path

    print(f"Saved {language}: {len(language_records)} -> {path}")

# 3. By-dataset/component files
dataset_paths = {}

for (dataset, language), selected in sorted(selected_components.items()):
    component_records = [
        record for record in benchmark_records
        if record["dataset"] == dataset
        and record["language"] == language
        and record["metadata"]["benchmark_component"] == dataset
    ]

    expected_n = TARGET_COUNTS[language][dataset]
    assert len(component_records) == expected_n, (dataset, language, len(component_records), expected_n)

    path = BY_DATASET_DIR / f"{dataset}_{language}_eval.jsonl"
    write_jsonl(component_records, path)
    dataset_paths[(dataset, language)] = path

    print(f"Saved {dataset}/{language}: {len(component_records)} -> {path}")

Saved full benchmark: 2500 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/benchmark_all.jsonl
Saved chinese: 500 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/by_language/benchmark_chinese.jsonl
Saved hindi: 500 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/by_language/benchmark_hindi.jsonl
Saved italian: 500 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/by_language/benchmark_italian.jsonl
Saved russian: 500 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/by_language/benchmark_russian.jsonl
Saved spanish: 500 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/by_language/benchmark_spanish.jsonl
Saved ceval/chinese: 150 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/by_dataset/ce

In [19]:
saved_paths = [ALL_PATH]
saved_paths += sorted(BY_LANGUAGE_DIR.glob("*.jsonl"))
saved_paths += sorted(BY_DATASET_DIR.glob("*.jsonl"))

EXPECTED_TOP_LEVEL_KEYS = TOP_LEVEL_ORDER

for path in saved_paths:
    loaded = read_jsonl(path)

    for record in loaded:
        assert list(record.keys()) == EXPECTED_TOP_LEVEL_KEYS, path
        assert record["source_id"] == get_source_id(record)
        assert record["split"] == "test"
        assert record["task_type"] == "mcq"
        assert record["answer"] in VALID_ANSWER_LETTERS
        assert f"{record['answer']}) " in record["prompt"]

print("All saved benchmark files have standardized field order and schema.")

# Partition checks
all_loaded = read_jsonl(ALL_PATH)
all_ids = {get_source_id(record) for record in all_loaded}

by_language_ids = set()
for path in sorted(BY_LANGUAGE_DIR.glob("*.jsonl")):
    by_language_ids.update(get_source_id(record) for record in read_jsonl(path))

by_dataset_ids = set()
for path in sorted(BY_DATASET_DIR.glob("*.jsonl")):
    by_dataset_ids.update(get_source_id(record) for record in read_jsonl(path))

assert all_ids == by_language_ids
assert all_ids == by_dataset_ids

print("Partition checks passed.")

All saved benchmark files have standardized field order and schema.
Partition checks passed.


In [20]:
manifest = {
    "benchmark_name": "cs552_multilingual_regional_mcqa",
    "version": "v1",
    "random_seed": RANDOM_SEED,
    "total_records": len(benchmark_records),
    "records_per_language": 500,
    "target_counts": TARGET_COUNTS,
    "files": {
        "all": str(ALL_PATH.relative_to(BENCHMARK_DIR)),
        "by_language": {
            language: str(path.relative_to(BENCHMARK_DIR))
            for language, path in sorted(language_paths.items())
        },
        "by_dataset": {
            f"{dataset}_{language}": str(path.relative_to(BENCHMARK_DIR))
            for (dataset, language), path in sorted(dataset_paths.items())
        },
    },
    "counts": {
        "by_language": {
            k: int(v)
            for k, v in bench_df["language"].value_counts().sort_index().items()
        },
        "by_dataset": {
            k: int(v)
            for k, v in bench_df["dataset"].value_counts().sort_index().items()
        },
        "by_language_dataset": {
            f"{language}/{dataset}": int(n)
            for (language, dataset), n in bench_df.groupby(["language", "dataset"]).size().items()
        },
        "by_language_answer": {
            f"{language}/{answer}": int(n)
            for (language, answer), n in bench_df.groupby(["language", "answer"]).size().items()
        },
    },
    "contamination_check": {
        "active_distilled_root": str(DISTILLED_ROOT),
        "active_distilled_files": len(active_distilled_paths),
        "unique_active_distilled_source_ids": len(distilled_ids),
        "overlap_with_active_distilled": 0,
        "backup_dirs_excluded": True,
    },
    "notes": [
        "benchmark_all.jsonl is the canonical evaluation file.",
        "by_language files are exact partitions of benchmark_all.jsonl.",
        "by_dataset files are exact partitions of benchmark_all.jsonl.",
        "All selected benchmark source IDs were checked against active *_distilled.jsonl files under v2_fallback, excluding backup_before_benchmark_filter_* directories.",
        "Most tasks use A-D answers. HEAD-QA may include five-option questions, so E can appear.",
    ],
}

with MANIFEST_PATH.open("w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(f"Saved manifest -> {MANIFEST_PATH}")

Saved manifest -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/manifest.json


In [21]:
print("Final benchmark folder:")

for path in sorted(BENCHMARK_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(BENCHMARK_DIR))

Final benchmark folder:
README.md
benchmark_all.jsonl
by_dataset/ceval_chinese_eval.jsonl
by_dataset/cmmlu_chinese_eval.jsonl
by_dataset/exams_italian_eval.jsonl
by_dataset/global_mmlu_chinese_eval.jsonl
by_dataset/global_mmlu_hindi_eval.jsonl
by_dataset/global_mmlu_italian_eval.jsonl
by_dataset/global_mmlu_russian_eval.jsonl
by_dataset/global_mmlu_spanish_eval.jsonl
by_dataset/head_qa_spanish_eval.jsonl
by_dataset/include_chinese_eval.jsonl
by_dataset/include_hindi_eval.jsonl
by_dataset/include_italian_eval.jsonl
by_dataset/include_russian_eval.jsonl
by_dataset/include_spanish_eval.jsonl
by_dataset/mera_russian_eval.jsonl
by_dataset/milu_hindi_eval.jsonl
by_language/benchmark_chinese.jsonl
by_language/benchmark_hindi.jsonl
by_language/benchmark_italian.jsonl
by_language/benchmark_russian.jsonl
by_language/benchmark_spanish.jsonl
manifest.json
old/benchmark_all.jsonl
small/benchmark_all_small.jsonl
small/by_language/benchmark_chinese_small.jsonl
small/by_language/benchmark_hindi_small.

# Small benchmark for fast eval of group model

In [22]:
SMALL_BENCHMARK_DIR = BENCHMARK_DIR / "small"
SMALL_BY_LANGUAGE_DIR = SMALL_BENCHMARK_DIR / "by_language"

SMALL_BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
SMALL_BY_LANGUAGE_DIR.mkdir(parents=True, exist_ok=True)

SMALL_ALL_PATH = SMALL_BENCHMARK_DIR / "benchmark_all_small.jsonl"

SMALL_TARGET_COUNTS = {
    language: {
        dataset: count // 5
        for dataset, count in dataset_counts.items()
    }
    for language, dataset_counts in TARGET_COUNTS.items()
}

print("Small benchmark target counts:")
for language, dataset_counts in SMALL_TARGET_COUNTS.items():
    print(language, dataset_counts, "total:", sum(dataset_counts.values()))

small_sum_by_lang = {
    language: sum(dataset_counts.values())
    for language, dataset_counts in SMALL_TARGET_COUNTS.items()
}

assert all(n == 100 for n in small_sum_by_lang.values())
assert sum(small_sum_by_lang.values()) == 500

Small benchmark target counts:
russian {'include': 20, 'global_mmlu': 40, 'mera': 40} total: 100
chinese {'include': 20, 'global_mmlu': 20, 'ceval': 30, 'cmmlu': 30} total: 100
hindi {'include': 20, 'global_mmlu': 40, 'milu': 40} total: 100
italian {'include': 20, 'global_mmlu': 40, 'exams': 40} total: 100
spanish {'include': 20, 'global_mmlu': 40, 'head_qa': 40} total: 100


In [23]:
SMALL_RANDOM_SEED = 142

small_records = []

for language, dataset_counts in SMALL_TARGET_COUNTS.items():
    for dataset, n in dataset_counts.items():
        component_records = [
            record for record in benchmark_records
            if record["language"] == language
            and record["dataset"] == dataset
            and record["metadata"]["benchmark_component"] == dataset
        ]

        assert len(component_records) == TARGET_COUNTS[language][dataset], (
            language,
            dataset,
            len(component_records),
            TARGET_COUNTS[language][dataset],
        )

        seed = SMALL_RANDOM_SEED + abs(hash((language, dataset))) % 10_000

        if dataset in STRATIFIED_DATASETS:
            selected = sample_subject_stratified(component_records, n, seed)
        else:
            selected = sample_uniform(component_records, n, seed)

        small_records.extend(selected)

        print(
            f"{language:8s} | {dataset:12s} | "
            f"selected {len(selected):3d} / full component {len(component_records):3d}"
        )

print("\nSmall benchmark total:", len(small_records))

small_source_ids = [record["source_id"] for record in small_records]
assert len(small_source_ids) == len(set(small_source_ids))
assert set(small_source_ids).issubset(set(record["source_id"] for record in benchmark_records))
assert len(small_records) == 500

russian  | include      | selected  20 / full component 100
russian  | global_mmlu  | selected  40 / full component 200
russian  | mera         | selected  40 / full component 200
chinese  | include      | selected  20 / full component 100
chinese  | global_mmlu  | selected  20 / full component 100
chinese  | ceval        | selected  30 / full component 150
chinese  | cmmlu        | selected  30 / full component 150
hindi    | include      | selected  20 / full component 100
hindi    | global_mmlu  | selected  40 / full component 200
hindi    | milu         | selected  40 / full component 200
italian  | include      | selected  20 / full component 100
italian  | global_mmlu  | selected  40 / full component 200
italian  | exams        | selected  40 / full component 200
spanish  | include      | selected  20 / full component 100
spanish  | global_mmlu  | selected  40 / full component 200
spanish  | head_qa      | selected  40 / full component 200

Small benchmark total: 500


In [24]:
small_df = pd.DataFrame(
    {
        "source_id": r["source_id"],
        "language": r["language"],
        "dataset": r["dataset"],
        "answer": r["answer"],
        "subject": record_subject(r),
        "regionality": r.get("metadata", {}).get("regionality", ""),
    }
    for r in small_records
)

print("Small benchmark by language:")
display(small_df["language"].value_counts().sort_index().to_frame("n"))

print("\nSmall benchmark by language and dataset:")
display(
    small_df.groupby(["language", "dataset"])
    .size()
    .reset_index(name="n")
    .pivot(index="language", columns="dataset", values="n")
    .fillna(0)
    .astype(int)
)

print("\nSmall benchmark by answer:")
display(
    small_df.groupby(["language", "answer"])
    .size()
    .reset_index(name="n")
    .pivot(index="language", columns="answer", values="n")
    .fillna(0)
    .astype(int)
)

print("\nSmall benchmark subjects/categories:")
display(
    small_df.groupby(["language", "dataset", "subject"])
    .size()
    .reset_index(name="n")
    .sort_values(["language", "dataset", "n"], ascending=[True, True, False])
    .head(100)
)

Small benchmark by language:


,n
language,
chinese,100
hindi,100
italian,100
russian,100
spanish,100



Small benchmark by language and dataset:


dataset,ceval,cmmlu,exams,global_mmlu,head_qa,include,mera,milu
language,,,,,,,,
chinese,30,30,0,20,0,20,0,0
hindi,0,0,0,40,0,20,0,40
italian,0,0,40,40,0,20,0,0
russian,0,0,0,40,0,20,40,0
spanish,0,0,0,40,40,20,0,0



Small benchmark by answer:


answer,A,B,C,D,E
language,,,,,
chinese,22,17,31,30,0
hindi,24,27,24,25,0
italian,25,25,27,23,0
russian,30,20,24,26,0
spanish,19,23,30,26,2



Small benchmark subjects/categories:


,language,dataset,subject,n
0,chinese,ceval,accountant,1
1,chinese,ceval,advanced_mathematics,1
2,chinese,ceval,art_studies,1
3,chinese,ceval,basic_medicine,1
4,chinese,ceval,business_administration,1
5,chinese,ceval,chinese_language_and_literature,1
6,chinese,ceval,civil_servant,1
7,chinese,ceval,clinical_medicine,1
8,chinese,ceval,college_chemistry,1
9,chinese,ceval,college_economics,1


In [25]:
write_jsonl(small_records, SMALL_ALL_PATH)
print(f"Saved small benchmark: {len(small_records)} -> {SMALL_ALL_PATH}")

small_language_paths = {}

for language in sorted(TARGET_COUNTS):
    language_records = [
        record for record in small_records
        if record["language"] == language
    ]

    assert len(language_records) == 100

    path = SMALL_BY_LANGUAGE_DIR / f"benchmark_{language}_small.jsonl"
    write_jsonl(language_records, path)
    small_language_paths[language] = path

    print(f"Saved small {language}: {len(language_records)} -> {path}")

Saved small benchmark: 500 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/small/benchmark_all_small.jsonl
Saved small chinese: 100 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/small/by_language/benchmark_chinese_small.jsonl
Saved small hindi: 100 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/small/by_language/benchmark_hindi_small.jsonl
Saved small italian: 100 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/small/by_language/benchmark_italian_small.jsonl
Saved small russian: 100 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/small/by_language/benchmark_russian_small.jsonl
Saved small spanish: 100 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/small/by_language/benchmark_spanish_small.jsonl


In [26]:
small_saved_paths = [SMALL_ALL_PATH] + sorted(SMALL_BY_LANGUAGE_DIR.glob("*.jsonl"))

full_ids = {record["source_id"] for record in benchmark_records}

for path in small_saved_paths:
    loaded = read_jsonl(path)

    source_ids = [record["source_id"] for record in loaded]
    assert len(source_ids) == len(set(source_ids)), f"Duplicate source ids in {path}"
    assert set(source_ids).issubset(full_ids), f"{path} is not a subset of the full benchmark"

    for record in loaded:
        assert list(record.keys()) == TOP_LEVEL_ORDER
        validate_basic_record(record)
        assert record["source_id"] not in distilled_ids
        assert record["split"] == "test"
        assert record["task_type"] == "mcq"
        assert record["answer"] in VALID_ANSWER_LETTERS
        assert f"{record['answer']}) " in record["prompt"]

    print(f"Validated {path.relative_to(BENCHMARK_DIR)}: {len(loaded)} records")

print("Small benchmark validation passed.")

Validated small/benchmark_all_small.jsonl: 500 records
Validated small/by_language/benchmark_chinese_small.jsonl: 100 records
Validated small/by_language/benchmark_hindi_small.jsonl: 100 records
Validated small/by_language/benchmark_italian_small.jsonl: 100 records
Validated small/by_language/benchmark_russian_small.jsonl: 100 records
Validated small/by_language/benchmark_spanish_small.jsonl: 100 records
Small benchmark validation passed.


In [27]:
with MANIFEST_PATH.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

manifest["small_benchmark"] = {
    "total_records": len(small_records),
    "records_per_language": 100,
    "random_seed": SMALL_RANDOM_SEED,
    "target_counts": SMALL_TARGET_COUNTS,
    "files": {
        "all": str(SMALL_ALL_PATH.relative_to(BENCHMARK_DIR)),
        "by_language": {
            language: str(path.relative_to(BENCHMARK_DIR))
            for language, path in sorted(small_language_paths.items())
        },
    },
    "notes": [
        "The small benchmark is a strict subset of benchmark_all.jsonl.",
        "It contains 1/5 of each full benchmark component.",
        "It is intended for fast debugging and checkpoint smoke tests.",
    ],
}

with MANIFEST_PATH.open("w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(f"Updated manifest -> {MANIFEST_PATH}")

Updated manifest -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/manifest.json


# Add MMLU-Prox

In [28]:
MMLU_PROX_DIR = BENCHMARK_DIR / "mmlu_prox"
MMLU_PROX_BY_LANGUAGE_DIR = MMLU_PROX_DIR / "by_language"
MMLU_PROX_DIR.mkdir(parents=True, exist_ok=True)
MMLU_PROX_BY_LANGUAGE_DIR.mkdir(parents=True, exist_ok=True)
MMLU_PROX_PROCESSED_DIR = PROCESSED_DIR / "mmlu_prox"
MMLU_PROX_LANGUAGES = ["russian", "chinese", "hindi", "italian", "spanish"]
MMLU_PROX_N_BASE_QUESTIONS = 200
MMLU_PROX_RANDOM_SEED = 242

MMLU_PROX_ALL_PATH = MMLU_PROX_DIR / "mmlu_prox_hard_1000.jsonl"

SUBJECT_WEIGHTS = {
    # Regional / cultural / society-oriented subjects
    "law": 1.6,
    "history": 1.6,
    "economics": 1.6,
    "business": 1.6,
    "philosophy": 1.6,
    "psychology": 1.6,
    "other": 1.6,

    # General hard-reasoning subjects; keep them, but do not prioritize them
    "math": 1.0,
    "physics": 1.0,
    "chemistry": 1.0,
    "engineering": 1.0,
    "computer science": 1.0,

    # Relevant to medicine / health / applied knowledge
    "biology": 1.4,
    "health": 1.4,
}

In [29]:
mmlu_prox_eval_by_language = {}

for language in MMLU_PROX_LANGUAGES:
    path = MMLU_PROX_PROCESSED_DIR / f"mmlu_prox_{language}_eval.jsonl"
    assert path.exists(), path

    records = read_jsonl(path)

    for record in records:
        validate_basic_record(record)

        assert record["dataset"] == "mmlu_prox"
        assert record["language"] == language
        assert record["split"] == "eval"
        assert "base_question_id" in record["metadata"]

    mmlu_prox_eval_by_language[language] = records

    print(f"{language:8s}: {len(records):5d} eval records")

russian :  1786 eval records
chinese :  1786 eval records
hindi   :  1786 eval records
italian :  1786 eval records
spanish :  1786 eval records


In [30]:
# Group MMLU-ProX eval records by base_question_id.
# Keep only base questions available in all five languages.

records_by_base_id = defaultdict(dict)

for language, records in mmlu_prox_eval_by_language.items():
    for record in records:
        base_id = record["metadata"]["base_question_id"]
        records_by_base_id[base_id][language] = record

complete_base_groups = {
    base_id: lang_records
    for base_id, lang_records in records_by_base_id.items()
    if set(lang_records.keys()) == set(MMLU_PROX_LANGUAGES)
}

print("Total base groups:", len(records_by_base_id))
print("Complete 5-language base groups:", len(complete_base_groups))

assert len(complete_base_groups) >= MMLU_PROX_N_BASE_QUESTIONS

Total base groups: 1786
Complete 5-language base groups: 1786


In [31]:
def weighted_sample_base_questions(complete_base_groups, n, seed):
    rng = random.Random(seed)

    remaining_base_ids = list(complete_base_groups.keys())
    selected_base_ids = []

    while remaining_base_ids and len(selected_base_ids) < n:
        weights = []

        for base_id in remaining_base_ids:
            lang_records = complete_base_groups[base_id]

            # Subjects should be aligned across languages, so one language is enough.
            subject = record_subject(lang_records["russian"])
            weight = SUBJECT_WEIGHTS.get(subject, 1.0)

            weights.append(weight)

        chosen_base_id = rng.choices(remaining_base_ids, weights=weights, k=1)[0]

        selected_base_ids.append(chosen_base_id)
        remaining_base_ids.remove(chosen_base_id)

    assert len(selected_base_ids) == n
    return selected_base_ids


selected_base_ids = weighted_sample_base_questions(
    complete_base_groups=complete_base_groups,
    n=MMLU_PROX_N_BASE_QUESTIONS,
    seed=MMLU_PROX_RANDOM_SEED,
)

print("Selected base questions:", len(selected_base_ids))

Selected base questions: 200


In [32]:
mmlu_prox_benchmark_records = []

for base_id in selected_base_ids:
    lang_records = complete_base_groups[base_id]

    for language in MMLU_PROX_LANGUAGES:
        record = dict(lang_records[language])

        if not isinstance(record.get("id"), str) or not record["id"].strip():
            record["id"] = get_source_id(record)

        record["split"] = "test"

        metadata = dict(record.get("metadata", {}))
        metadata["benchmark_name"] = "cs552_multilingual_regional_mcqa"
        metadata["benchmark_version"] = "mmlu_prox_hard_v1"
        metadata["benchmark_component"] = "mmlu_prox"
        metadata["benchmark_language"] = language
        metadata["benchmark_source_id"] = get_source_id(record)
        metadata["benchmark_base_question_id"] = base_id

        record["metadata"] = metadata

        standardized = standardize_benchmark_record(record)
        mmlu_prox_benchmark_records.append(standardized)

print("MMLU-ProX hard benchmark records:", len(mmlu_prox_benchmark_records))

assert len(mmlu_prox_benchmark_records) == MMLU_PROX_N_BASE_QUESTIONS * len(MMLU_PROX_LANGUAGES)
assert len(mmlu_prox_benchmark_records) == 1000

MMLU-ProX hard benchmark records: 1000


In [33]:
mmlu_prox_df = pd.DataFrame(
    {
        "source_id": r["source_id"],
        "base_question_id": r["metadata"]["benchmark_base_question_id"],
        "language": r["language"],
        "dataset": r["dataset"],
        "answer": r["answer"],
        "subject": record_subject(r),
        "n_options": r["metadata"].get("n_options"),
    }
    for r in mmlu_prox_benchmark_records
)

print("By language:")
display(mmlu_prox_df["language"].value_counts().sort_index().to_frame("n"))

print("\nBy subject:")
display(
    mmlu_prox_df.groupby(["language", "subject"])
    .size()
    .reset_index(name="n")
    .sort_values(["language", "subject"])
)

print("\nBy answer:")
display(
    mmlu_prox_df.groupby(["language", "answer"])
    .size()
    .reset_index(name="n")
    .pivot(index="language", columns="answer", values="n")
    .fillna(0)
    .astype(int)
)

print("\nBy n_options:")
display(
    mmlu_prox_df.groupby(["language", "n_options"])
    .size()
    .reset_index(name="n")
    .pivot(index="language", columns="n_options", values="n")
    .fillna(0)
    .astype(int)
)

print("\nUnique base questions:", mmlu_prox_df["base_question_id"].nunique())

assert mmlu_prox_df["language"].value_counts().eq(MMLU_PROX_N_BASE_QUESTIONS).all()
assert mmlu_prox_df["base_question_id"].nunique() == MMLU_PROX_N_BASE_QUESTIONS

By language:


,n
language,
chinese,200
hindi,200
italian,200
russian,200
spanish,200



By subject:


,language,subject,n
0,chinese,biology,9
1,chinese,business,17
2,chinese,chemistry,16
3,chinese,computer science,2
4,chinese,economics,14
5,chinese,engineering,17
6,chinese,health,9
7,chinese,history,7
8,chinese,law,19
9,chinese,math,13



By answer:


answer,A,B,C,D,E,F,G,H,I,J
language,,,,,,,,,,
chinese,23,23,23,32,14,17,18,20,16,14
hindi,23,23,23,32,14,17,18,20,16,14
italian,23,23,23,32,14,17,18,20,16,14
russian,23,23,23,32,14,17,18,20,16,14
spanish,23,23,23,32,14,17,18,20,16,14



By n_options:


n_options,4,5,7,8,9,10
language,,,,,,
chinese,7,2,7,4,12,168
hindi,7,2,7,4,12,168
italian,7,2,7,4,12,168
russian,7,2,7,4,12,168
spanish,7,2,7,4,12,168



Unique base questions: 200


In [34]:
write_jsonl(mmlu_prox_benchmark_records, MMLU_PROX_ALL_PATH)
print(f"Saved MMLU-ProX hard benchmark: {len(mmlu_prox_benchmark_records)} -> {MMLU_PROX_ALL_PATH}")

mmlu_prox_language_paths = {}

for language in MMLU_PROX_LANGUAGES:
    language_records = [
        record for record in mmlu_prox_benchmark_records
        if record["language"] == language
    ]

    assert len(language_records) == MMLU_PROX_N_BASE_QUESTIONS

    path = MMLU_PROX_BY_LANGUAGE_DIR / f"mmlu_prox_{language}_hard_200.jsonl"
    write_jsonl(language_records, path)
    mmlu_prox_language_paths[language] = path

    print(f"Saved MMLU-ProX {language}: {len(language_records)} -> {path}")

Saved MMLU-ProX hard benchmark: 1000 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/mmlu_prox/mmlu_prox_hard_1000.jsonl
Saved MMLU-ProX russian: 200 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/mmlu_prox/by_language/mmlu_prox_russian_hard_200.jsonl
Saved MMLU-ProX chinese: 200 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/mmlu_prox/by_language/mmlu_prox_chinese_hard_200.jsonl
Saved MMLU-ProX hindi: 200 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/mmlu_prox/by_language/mmlu_prox_hindi_hard_200.jsonl
Saved MMLU-ProX italian: 200 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/mmlu_prox/by_language/mmlu_prox_italian_hard_200.jsonl
Saved MMLU-ProX spanish: 200 -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/mmlu_prox/by_l

In [35]:
mmlu_prox_saved_paths = [MMLU_PROX_ALL_PATH] + sorted(MMLU_PROX_BY_LANGUAGE_DIR.glob("*.jsonl"))

for path in mmlu_prox_saved_paths:
    loaded = read_jsonl(path)

    source_ids = [record["source_id"] for record in loaded]
    assert len(source_ids) == len(set(source_ids)), f"Duplicate source ids in {path}"

    for record in loaded:
        assert list(record.keys()) == TOP_LEVEL_ORDER
        validate_basic_record(record)
        assert record["dataset"] == "mmlu_prox"
        assert record["split"] == "test"
        assert record["task_type"] == "mcq"
        assert record["answer"] in VALID_ANSWER_LETTERS
        assert f"{record['answer']}) " in record["prompt"]
        assert "benchmark_base_question_id" in record["metadata"]

    print(f"Validated {path.relative_to(BENCHMARK_DIR)}: {len(loaded)} records")

print("MMLU-ProX hard benchmark validation passed.")

Validated mmlu_prox/mmlu_prox_hard_1000.jsonl: 1000 records
Validated mmlu_prox/by_language/mmlu_prox_chinese_hard_200.jsonl: 200 records
Validated mmlu_prox/by_language/mmlu_prox_hindi_hard_200.jsonl: 200 records
Validated mmlu_prox/by_language/mmlu_prox_italian_hard_200.jsonl: 200 records
Validated mmlu_prox/by_language/mmlu_prox_russian_hard_200.jsonl: 200 records
Validated mmlu_prox/by_language/mmlu_prox_spanish_hard_200.jsonl: 200 records
MMLU-ProX hard benchmark validation passed.


In [36]:
with MANIFEST_PATH.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

manifest["mmlu_prox_hard_benchmark"] = {
    "total_records": len(mmlu_prox_benchmark_records),
    "records_per_language": MMLU_PROX_N_BASE_QUESTIONS,
    "unique_base_questions": MMLU_PROX_N_BASE_QUESTIONS,
    "random_seed": MMLU_PROX_RANDOM_SEED,
    "subject_weights": SUBJECT_WEIGHTS,
    "files": {
        "all": str(MMLU_PROX_ALL_PATH.relative_to(BENCHMARK_DIR)),
        "by_language": {
            language: str(path.relative_to(BENCHMARK_DIR))
            for language, path in sorted(mmlu_prox_language_paths.items())
        },
    },
    "notes": [
        "MMLU-ProX hard benchmark is separate from benchmark_all.jsonl.",
        "It is sampled only from the held-out MMLU-ProX eval split.",
        "The same base_question_id values are used across all five languages for fair cross-language comparison.",
        "Sampling is weighted toward regional, cultural, society-oriented, health, and biology subjects.",
        "MMLU-ProX may include variable numbers of answer options, up to A-J.",
    ],
}

with MANIFEST_PATH.open("w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print(f"Updated manifest -> {MANIFEST_PATH}")

Updated manifest -> /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/benchmarks/manifest.json
